In [1]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "GPU memory (GB):",
        round(
            torch.cuda.get_device_properties(0).total_memory
            / (1024**3),
            2,
        ),
    )
else:
    print("No GPU allocated; CPU runtime is active.")

PyTorch: 2.11.0+cu130
CUDA available: True
GPU: Tesla T4
GPU memory (GB): 14.56


## 1.load the dataset

In [2]:

# Import libraries for file handling and data preparation.
from pathlib import Path
import zipfile

from google.colab import files

# Upload the ZIP containing data/quality_assessment/.
uploaded = files.upload()

# Find the uploaded ZIP file.
zip_files = [
    name for name in uploaded
    if name.lower().endswith(".zip")
]

if not zip_files:
    raise ValueError("Please upload a .zip file containing the dataset.")

zip_path = Path("/content") / zip_files[0]

# Extract into /content/project so that the original relative
# paths such as data/quality_assessment/original/... still work.
PROJECT_ROOT = Path("/content/project")
PROJECT_ROOT.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(zip_path, "r") as archive:
    archive.extractall(PROJECT_ROOT)

# Define paths to the prepared CSV files.
DATA_DIR = PROJECT_ROOT / "data" / "quality_assessment"
TRAIN_CSV = DATA_DIR / "processed" / "quality_train.csv"
TEST_CSV = DATA_DIR / "processed" / "quality_test.csv"

# Confirm that extraction worked.
print("Project directory:", PROJECT_ROOT)
print("Training CSV exists:", TRAIN_CSV.exists())
print("Test CSV exists:", TEST_CSV.exists())

if not TRAIN_CSV.exists() or not TEST_CSV.exists():
    raise FileNotFoundError(
        "The ZIP must preserve the folder structure "
        "data/quality_assessment/processed/quality_train.csv "
        "and quality_test.csv."
    )


Saving data.zip to data.zip
Project directory: /content/project
Training CSV exists: True
Test CSV exists: True


## 2.check the training environment

In [3]:

# Check the GPU and the deep-learning libraries before training.
import torch
import torchvision
import pandas as pd

print("PyTorch:", torch.__version__)
print("Torchvision:", torchvision.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
    print("GPU:", torch.cuda.get_device_name(0))
else:
    DEVICE = torch.device("cpu")
    print("Using CPU.")




PyTorch: 2.11.0+cu130
Torchvision: 0.26.0+cu130
CUDA available: True
GPU: Tesla T4


In [4]:
# Load the CSV files.
train_df = pd.read_csv(TRAIN_CSV)
test_df = pd.read_csv(TEST_CSV)

print("\nTraining records:", len(train_df))
print("Test records:", len(test_df))

print("\nTraining labels:")
print(train_df["General_visual_Quality"].value_counts())

print("\nTest labels:")
print(test_df["General_visual_Quality"].value_counts())


Training records: 27
Test records: 9

Training labels:
General_visual_Quality
Acceptable        19
Not Acceptable     8
Name: count, dtype: int64

Test labels:
General_visual_Quality
Acceptable        8
Not Acceptable    1
Name: count, dtype: int64


In [5]:
# Check that the expected columns exist.
required_columns = {
    "Sample_ID",
    "Relative_Path",
    "General_visual_Quality",
}

for name, df in [("train", train_df), ("test", test_df)]:
    missing = required_columns - set(df.columns)
    if missing:
        raise ValueError(f"{name} CSV is missing columns: {missing}")

In [6]:
# Ensure no physical plant sample appears in both splits.
overlap = set(train_df["Sample_ID"]) & set(test_df["Sample_ID"])
print("\nSample overlap:", len(overlap))

if overlap:
    raise ValueError("Data leakage detected: samples overlap.")


Sample overlap: 0


## **3. Build the image dataset and transformations**

This cell loads the images and applies light augmentation to training images only. It uses the repository-relative image paths, not the old Windows paths.

In [7]:
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

# Use a consistent class order so model outputs remain interpretable.
CLASS_NAMES = ["Acceptable", "Not Acceptable"]
CLASS_TO_IDX = {name: idx for idx, name in enumerate(CLASS_NAMES)}

In [8]:
# ResNet18 expects normalized RGB images.
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=10),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225],
    ),
])

# Do not augment test images: evaluate them consistently.
test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225],
    ),
])


class PlantQualityDataset(Dataset):
    def __init__(self, dataframe, transform):
        self.df = dataframe.reset_index(drop=True)
        self.transform = transform

        # Reject unknown labels rather than silently mislabelling them.
        unknown = set(self.df["General_visual_Quality"].dropna()) - set(CLASS_NAMES)
        if unknown:
            raise ValueError(f"Unexpected labels: {unknown}")

    def __len__(self):
        return len(self.df)

    def __getitem__(self, index):
        row = self.df.iloc[index]

        # Reconstruct the image path inside the Colab project directory.
        image_path = PROJECT_ROOT / str(row["Relative_Path"])

        if not image_path.is_file():
            raise FileNotFoundError(f"Image not found: {image_path}")

        # Convert all images to RGB, even if an original is grayscale.
        image = Image.open(image_path).convert("RGB")
        image = self.transform(image)

        label = CLASS_TO_IDX[row["General_visual_Quality"]]
        return image, label


train_dataset = PlantQualityDataset(train_df, train_transform)
test_dataset = PlantQualityDataset(test_df, test_transform)

# Use a small batch size suitable for the available dataset.
train_loader = DataLoader(
    train_dataset, batch_size=8, shuffle=True, num_workers=2
)
test_loader = DataLoader(
    test_dataset, batch_size=8, shuffle=False, num_workers=2
)

print("Training batches:", len(train_loader))
print("Test batches:", len(test_loader))

# Read one batch to confirm images load correctly.
images, labels = next(iter(train_loader))
print("Image tensor shape:", images.shape)
print("Label tensor shape:", labels.shape)

Training batches: 4
Test batches: 2
Image tensor shape: torch.Size([8, 3, 224, 224])
Label tensor shape: torch.Size([8])


# 4.Train the model using transfer learning

Start from ResNet18 weights pretrained on ImageNet, freeze most of the network, and train its final feature block and classification layer. This is a reasonable starting experiment for a small dataset.

In [9]:

import copy
import torch.nn as nn
from torchvision.models import resnet18, ResNet18_Weights

# Download pretrained weights if they are not already cached.
model = resnet18(weights=ResNet18_Weights.DEFAULT)

# Freeze all pretrained parameters first.
for parameter in model.parameters():
    parameter.requires_grad = False

# Fine-tune the final residual block.
for parameter in model.layer4.parameters():
    parameter.requires_grad = True

# Replace the original 1000-class output layer with our two classes.
input_features = model.fc.in_features
model.fc = nn.Linear(input_features, len(CLASS_NAMES))

model = model.to(DEVICE)

# Give more weight to the minority class in the training loss.
# This can help with imbalance, but does not create new examples.
counts = (
    train_df["General_visual_Quality"]
    .value_counts()
    .reindex(CLASS_NAMES, fill_value=0)
)

if (counts == 0).any():
    raise ValueError(
        "Both classes must be represented in the training data."
    )

class_weights = len(train_df) / (
    len(CLASS_NAMES) * torch.tensor(counts.values, dtype=torch.float32)
)
criterion = nn.CrossEntropyLoss(
    weight=class_weights.to(DEVICE)
)

# Update only the unfrozen parameters.
optimizer = torch.optim.AdamW(
    [p for p in model.parameters() if p.requires_grad],
    lr=0.0001,
    weight_decay=0.01,
)

# Use a fixed seed for more reproducible experiments.
torch.manual_seed(42)

EPOCHS = 8
best_loss = float("inf")
best_state = copy.deepcopy(model.state_dict())

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0.0
    correct = 0
    total = 0

    for images, labels in train_loader:
        images = images.to(DEVICE)
        labels = labels.to(DEVICE)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item() * labels.size(0)
        correct += (outputs.argmax(dim=1) == labels).sum().item()
        total += labels.size(0)

    epoch_loss = total_loss / total
    epoch_accuracy = correct / total

    print(
        f"Epoch {epoch + 1}/{EPOCHS} | "
        f"Training loss: {epoch_loss:.4f} | "
        f"Training accuracy: {epoch_accuracy:.3f}"
    )

    # Keep the best training-loss checkpoint for this initial experiment.
    # This is not a substitute for validation-based model selection.
    if epoch_loss < best_loss:
        best_loss = epoch_loss
        best_state = copy.deepcopy(model.state_dict())

model.load_state_dict(best_state)
print("\nTraining finished.")


Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 162MB/s]


Epoch 1/8 | Training loss: 0.7396 | Training accuracy: 0.704
Epoch 2/8 | Training loss: 0.4521 | Training accuracy: 0.741
Epoch 3/8 | Training loss: 0.4763 | Training accuracy: 0.704
Epoch 4/8 | Training loss: 0.2894 | Training accuracy: 0.852
Epoch 5/8 | Training loss: 0.3428 | Training accuracy: 0.889
Epoch 6/8 | Training loss: 0.0971 | Training accuracy: 0.963
Epoch 7/8 | Training loss: 0.1790 | Training accuracy: 0.963
Epoch 8/8 | Training loss: 0.2023 | Training accuracy: 0.963

Training finished.


## 5.Evaluate on the held-out test set

In [10]:

import numpy as np
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
)

# Switch to evaluation mode and disable gradient calculation.
model.eval()
true_labels = []
predicted_labels = []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(DEVICE)

        # Obtain the predicted class for each image.
        outputs = model(images)
        predictions = outputs.argmax(dim=1).cpu().numpy()

        predicted_labels.extend(predictions.tolist())
        true_labels.extend(labels.numpy().tolist())

# Report performance. Interpret these metrics cautiously because
# the test set contains only nine images from five plant samples.
print("Test accuracy:", round(
    accuracy_score(true_labels, predicted_labels), 4
))

print("\nConfusion matrix:")
print(confusion_matrix(
    true_labels,
    predicted_labels,
    labels=[0, 1],
))

print("\nClassification report:")
print(classification_report(
    true_labels,
    predicted_labels,
    labels=[0, 1],
    target_names=CLASS_NAMES,
    zero_division=0,
))

# Keep the raw predictions for later inspection.
evaluation_df = test_df[["Sample_ID", "Relative_Path",
                         "General_visual_Quality"]].copy()
evaluation_df["Predicted_Quality"] = [
    CLASS_NAMES[idx] for idx in predicted_labels
]

print("\nIndividual predictions:")
display(evaluation_df)


Test accuracy: 0.7778

Confusion matrix:
[[7 1]
 [1 0]]

Classification report:
                precision    recall  f1-score   support

    Acceptable       0.88      0.88      0.88         8
Not Acceptable       0.00      0.00      0.00         1

      accuracy                           0.78         9
     macro avg       0.44      0.44      0.44         9
  weighted avg       0.78      0.78      0.78         9


Individual predictions:


,Sample_ID,Relative_Path,General_visual_Quality,Predicted_Quality
0,S001,data/quality_assessment/original/individual/P0...,Acceptable,Acceptable
1,S001,data/quality_assessment/original/individual/P0...,Acceptable,Acceptable
2,S002,data/quality_assessment/original/individual/P0...,Acceptable,Acceptable
3,S002,data/quality_assessment/original/individual/P0...,Acceptable,Acceptable
4,S006,data/quality_assessment/original/individual/P0...,Acceptable,Acceptable
5,S006,data/quality_assessment/original/individual/P0...,Acceptable,Not Acceptable
6,S009,data/quality_assessment/original/individual/P0...,Acceptable,Acceptable
7,S009,data/quality_assessment/original/individual/P0...,Acceptable,Acceptable
8,S012,data/quality_assessment/original/individual/P0...,Not Acceptable,Acceptable


## 6. Save the trained model

In [11]:

import json

# Store model weights and the metadata needed to interpret outputs.
OUTPUT_DIR = Path("/content/model_artifacts")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MODEL_PATH = OUTPUT_DIR / "quality_resnet18.pth"
METADATA_PATH = OUTPUT_DIR / "quality_model_metadata.json"

torch.save(
    {
        "model_state_dict": model.state_dict(),
        "class_names": CLASS_NAMES,
        "architecture": "resnet18",
        "image_size": 224,
    },
    MODEL_PATH,
)

metadata = {
    "architecture": "resnet18",
    "classes": CLASS_NAMES,
    "image_size": 224,
    "training_records": len(train_df),
    "test_records": len(test_df),
    "note": (
        "Initial proof-of-concept model. "
        "Evaluation is preliminary because the labelled dataset is small."
    ),
}

METADATA_PATH.write_text(json.dumps(metadata, indent=2))

print("Saved model:", MODEL_PATH)
print("Saved metadata:", METADATA_PATH)

# Download the files to your computer.
from google.colab import files
files.download(str(MODEL_PATH))
files.download(str(METADATA_PATH))


Saved model: /content/model_artifacts/quality_resnet18.pth
Saved metadata: /content/model_artifacts/quality_model_metadata.json


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>